# Install and check the custom OGS HM toolkit

This notebook installs `ogs-hm-tools`, uses upstream OGSTools, and runs a small
Mohr–Coulomb HM installation example. It needs no development repository
or previous solver results. The supplied wheel is a **private prototype for Linux
x86_64 and Python 3.12**. Windows users need a compatible Linux/WSL environment.
A Python virtual environment isolates Python packages; this is not a clean-OS test.

Place the wheel beside this notebook, or set `OGS_HM_WHEEL` to its location.
Inside the toolkit checkout the cell also finds `../dist/`. A public release URL
will replace this local file selection once packaging and license checks finish.
Internet access is required for upstream Python dependencies. Use a fresh kernel;
restart it after installation if any of these packages were already imported.

In [ ]:
import os, platform, sys, subprocess
from pathlib import Path
assert platform.system() == "Linux" and platform.machine() == "x86_64", "Use Linux x86_64 for this prototype."
assert sys.version_info[:2] == (3, 12), "Select a Python 3.12 kernel."
configured = os.environ.get("OGS_HM_WHEEL")
if configured:
    wheel = Path(configured).expanduser().resolve()
else:
    candidates = sorted({p.resolve() for folder in [Path.cwd(), Path.cwd()/"dist", Path.cwd().parent/"dist"]
                         for p in folder.glob("ogs_hm_tools-0.1.0.dev0-cp312-cp312-linux_x86_64.whl")})
    if len(candidates) != 1:
        raise FileNotFoundError("Place the supplied wheel beside the notebook, or set OGS_HM_WHEEL.")
    wheel = candidates[0]
assert wheel.is_file(), wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", str(wheel)])

In [ ]:
import json
import ogstools as ot
import ogs_hm_tools as hm
print(json.dumps(hm.doctor(), indent=2))

## Run the supplied MC column

The one-metre saturated column uses Pa, m and s, gravity 10 m/s², fluid density
1000 kg/m³ and reservoir head 1.3 m. A synchronous numerical initialisation ramp
reaches full load at t=1, followed by a hold through t=2. Pressure is prescribed
on primary nodes throughout the column: this demonstrates installation and native
HM phase syntax, not boundary-driven seepage. The example uses the previously
qualified MC material; no strength or convergence controls are changed. It is
a derivative of stage 04a: the obsolete fluid bulk_modulus property is removed,
so Constant fluid density now explicitly means zero fluid compressibility.
This installation check does not reproduce the historical fluid-reaction test.

Each execution creates a fresh work folder. `hm.run` selects the custom solver
explicitly; stock OGS utilities and non-HM notebooks remain available separately.

In [ ]:
from tempfile import mkdtemp
work = Path(mkdtemp(prefix="ogs-hm-notebook-", dir=Path.cwd()))
project = hm.example(work/"column")
output = hm.run(project, work/"output")
series = ot.MeshSeries(output/"strict_esa.pvd")
print("Saved states:", len(series.timevalues), "Final time:", series.timevalues[-1])
print("Work folder:", work)

## Check the calculated fields

Check the hydrostatic pressure at primary FE nodes at every saved time, the
initialisation and final endpoints, and unchanged displacement/stress during the
full-load hold. Higher-order displacement nodes do not carry primary pressure
DOFs. These checks cover installation and a known numerical case. The development
study separately checks weak force balance, fluid inventory and MC history;
this small notebook is not a general solver or material qualification.

In [ ]:
import xml.etree.ElementTree as ET
import meshio
import numpy as np
rows = ET.parse(output/"strict_esa.pvd").findall(".//DataSet")
frames = [(float(row.get("timestep")), meshio.read(output/row.get("file"))) for row in rows]
assert frames and abs(frames[-1][0] - 2) < 1e-12
worst_pressure = 0.0
for time, mesh in frames:
    primary = np.unique(mesh.cells_dict["triangle6"][:, :3])
    target = min(time, 1.0)*10000*(1.3-mesh.points[primary, 1])
    error = np.max(np.abs(mesh.point_data["pressure"].ravel()[primary]-target))
    worst_pressure = max(worst_pressure, float(error))
assert worst_pressure < 1e-6, worst_pressure
ramp_end = next(mesh for time, mesh in frames if abs(time-1) < 1e-12)
final = frames[-1][1]
hold_u = float(np.max(np.abs(final.point_data["displacement"]-ramp_end.point_data["displacement"])))
hold_sigma = float(np.max(np.abs(final.field_data["sigma_ip"]-ramp_end.field_data["sigma_ip"])))
assert hold_u < 1e-8 and hold_sigma < 0.1, (hold_u, hold_sigma)
check = dict(passed=True, saved_states=len(frames), pressure_error_Pa=worst_pressure,
             hold_displacement_change_m=hold_u, hold_stress_change_Pa=hold_sigma)
(work/"installation_check.json").write_text(json.dumps(check, indent=2))
print(json.dumps(check, indent=2))